# DerStandard Forum API Scraper

DerStandard loads comments via a **JSON API** — no Playwright or HTML parsing needed.

**Discovered endpoint:**
```
https://apps.derstandard.at/forum/1/{ARTICLE_ID}
```
The article ID is the number in every DerStandard story URL:
```
https://www.derstandard.at/story/2000135466855/titel-des-artikels
                                   ^^^^^^^^^^^^^
                                   this is your article ID
```

**What this scraper collects:**
- Comment text only (no usernames — GDPR-conscious)
- Comment timestamp
- Parent comment ID (thread structure)
- Up/downvote counts (from second endpoint)
- Article metadata (title, section, date) from the article URL itself

**Legal basis:** EU DSM Directive 2019/790 Art. 3 TDM exception for scientific research at a university.
Do not publish raw comment text. Use for analysis only.

---
**Input:** A list of DerStandard article URLs containing your keywords (Kindeswohl, Entfremdung, Sorgerecht...)
You find these via Google: `site:derstandard.at Kindeswohl Sorgerecht`

**Output:** `data/derstandard_comments.json`

In [25]:
!pip install playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 18.6/43.2 MB 94.6 kB/s  0:04:20m
Resuming download playwright-1.59.0-py3-none-macosx_11_0_universal2.whl (18.6 MB/43.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 36.7/43.2 MB 123.2 kB/s eta 0:00:53
error: incomplete-download

× Download failed after 6 attempts because not enough bytes were received (36.7 MB/43.2 MB)
╰─> URL: https://files.pythonhosted.org/packages/80/91/fd219aa78ca03d37e93aaedaed4e224131e3090a9264f9bb773c8271d67e/playwright-1.59.0-py3-none-macosx_11_0_universal2.whl

note: This is an issue with network connectivity, not pip.
hint: Use --resume-retries to configure resume attempt limit.
zsh:1: command not found: playwright


In [17]:
import requests
import json
import time
import re
from datetime import datetime
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

# Polite scraping headers — identify your project
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; ThesisResearch/1.0; academic NLP research)',
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://www.derstandard.at/',
    'Cookie': "_pcid=%7B%22browserId%22%3A%22mms7txh3pz6x05ys%22%7D; xbc=%7Bkpcd%7DChBtbXM3dHhoM3B6NngwNXlzEgpJQVIwY1ZBMnB1GjxQSjdiMUJKUWVMWHpOWGlhaHpRTFR5RVlrOEFqWmFUNG1LQVBjSDJqcmIxN3NsRThvOVlKdGVIU05QMDEgAA; DSGVO_ZUSAGE_V1=true; MGUID=GUID=a8da63ca-b50e-4ea6-94f9-44338ceeec39&Timestamp=2026-03-15T20:37:08&DetectedVersion=&Version=&BIV=2&Hash=0A9C1B96D3CD827BA697F8F1BAB4E634; _ga=GA1.1.1594777500.1773607030; FPID=FPID2.2.eG7U1N5rQxNojUuchG9BbVnjw1CZJa6hssiz%2FR9Dh4c%3D.1773607030; MGUIDBAK=GUID=a8da63ca-b50e-4ea6-94f9-44338ceeec39&Timestamp=2026-03-15T20:37:08&DetectedVersion=&Version=&BIV=2&Hash=0A9C1B96D3CD827BA697F8F1BAB4E634; _ga_JD7GJ3PL6J=GS2.1.s1773607029$o1$g0$t1773607036$j53$l0$h1345917106; _sp_su=false; tcfs=1; consentUUID=758e3587-ab64-4359-9e41-9900b2fb8ad3_56; consentDate=2026-05-10T12:13:12.301Z; __pat=7200000; cX_P=mms7txh3pz6x05ys; FPLC=LFkfzqa%2BTLkGTGWPY%2B5BV9THXgrww7tgDm6pdF7lKOS3cc1ah4Kpl3bJP%2BNUSXjbdqainYGi1PRlZk4tV0SMeluFuAPlLTgALvLEz9B7NqdzjXj0igQj%2Fr%2FhxLk3dQ%3D%3D; __RequestVerificationToken_L2ZvcnVt0=bHvk2AC0rZDV9Cf1ITgS0XoT_kaTtL_KIh2s6lLeaRZylgcsHKU4XN6E33p6S5Eob1xrUq1Ej_TafJw5FES-l8WjCcc1; ioam2018=19820511751965e7d2de06a00765d%3A1809519198046%3A1778415198046%3A.derstandard.at%3A5%3Aat_w_atderstand%3ARedCont%2FNachrichten%2FSonstiges%2Fmoewa%2F%3Anoevent%3A1778416165236%3A3wqt6d; __pvi=eyJpZCI6InYtbW96cWh1eGdjZWE5Y3VobyIsImRvbWFpbiI6Ii5kZXJzdGFuZGFyZC5hdCIsInRpbWUiOjE3Nzg0MTYxNjU0OTZ9; FPGSID=1.1778415213.1778416166.G-TQ3BNDRZZ9.SAA5_t3MV6et8I502ypmJQ; __tbc=%7Bkpcd%7DChBtbXM3dHhoM3B6NngwNXlzEgpJQVIwY1ZBMnB1GjxQSjdiMUJKUWVMWHpOWGlhaHpRTFR5RVlrOEFqWmFUNG1LQVBjSDJqcmIxN3NsRThvOVlKdGVIU05QMDEgAA; _ga_TQ3BNDRZZ9=GS2.1.s1778415212$o1$g1$t1778416168$j53$l0$h569126246; _pctx=%7Bu%7DN4IgrgzgpgThIC5QEMAcATZA2AzAY2QFoAjAVgAYpCAWKbQgTmoDMGbqcdU8pe8cGwPAGsAlgF9EoAA4wozUQA9EIEaJAAaEABcAntKgqAwgA0Q48VsiwAytuTbIK5ADsA9i80gIo7VACS6CoATNQA7KjUHNQModQAjFjxCcE45kA"
}

RATE_LIMIT_SECONDS = 3  # be polite — 3 seconds between requests
print('Setup complete.')

Setup complete.


In [18]:
# ── Article URLs to scrape ─────────────────────────────────────────────────────
# Find these via: Google search  →  site:derstandard.at Kindeswohl Sorgerecht
# Or derstandard.at search for your keywords, then copy article URLs.
# Format: just paste the full URL — the ID is extracted automatically.

ARTICLE_URLS = [
    'https://www.derstandard.at/story/2000129268290/parental-alienation-syndrom-oder-wie-kinder-zu-opfern-ihrer-eltern',
    'https://www.derstandard.at/story/2000123917902/zur-kindeswohlpruefung-verpflichten',
    'https://www.derstandard.at/story/2000092116447/kopftuch-und-kindeswohl',
    'https://www.derstandard.at/story/2000142962434/kindeswohl-nicht-ausreichendvfgh-hob-abschiebeentscheidung-auf-weil',
    "https://www.derstandard.at/story/3000000319323/24-stunden-im-leben-einer-alleinerzieherin",
    "https://www.derstandard.at/story/2000135466855/allein-mit-kind-und-noeten-wir-haben-um-jeden-cent"
]

print(f'Articles to scrape: {len(ARTICLE_URLS)}')

Articles to scrape: 6


In [19]:
# ── Helper functions ───────────────────────────────────────────────────────────

def extract_article_id(url):
    """Extract the numeric article ID from a DerStandard story URL."""
    match = re.search(r'/story/(\d+)', url)
    return match.group(1) if match else None


def fetch_comments(article_id: str, timeout: int = 15) -> dict:
    """
    Fetch comments from the DerStandard forum API.
    Returns the raw JSON response as a dict, or an error dict.
    """
    url = f'https://apps.derstandard.at/forum/1/{article_id}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return {'ok': True, 'data': r.json(), 'url': url}
    except requests.exceptions.HTTPError as e:
        return {'ok': False, 'error': f'HTTP {r.status_code}', 'url': url}
    except Exception as e:
        return {'ok': False, 'error': str(e), 'url': url}


def fetch_ratings(article_id: str, timeout: int = 10) -> dict:
    """Fetch upvote/downvote ratings for the article's comments."""
    url = f'https://apps.derstandard.at/forum/1/rating?objectId={article_id}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except:
        return {}


def parse_comments(raw, article_id, article_url):
    """
    Flatten the comment tree into a list of records.
    Keeps: text, timestamp, comment_id, parent_id, thread depth.
    Drops: usernames (GDPR-conscious).
    
    NOTE: This function assumes a typical DerStandard API structure.
    Run the exploration cell first to verify the actual field names.
    """
    records = []
    
    # DerStandard API typically returns a list of top-level postings
    # each with nested 'children' for replies
    # We'll handle both flat and nested structures
    
    def process_posting(posting: dict, depth: int = 0, parent_id=None):
        record = {
            'article_id':  article_id,
            'article_url': article_url,
            'comment_id':  posting.get('id') or posting.get('postingId'),
            'parent_id':   parent_id,
            'depth':       depth,
            'text':        posting.get('text') or posting.get('body') or posting.get('content', ''),
            'timestamp':   posting.get('publishedDate') or posting.get('createdAt') or posting.get('date', ''),
            'upvotes':     posting.get('positiveVotes') or posting.get('upvotes') or posting.get('likes', 0),
            'downvotes':   posting.get('negativeVotes') or posting.get('downvotes', 0),
            'is_deleted':  posting.get('isDeleted', False),
        }
        records.append(record)
        
        # Recurse into children/replies
        children = posting.get('children') or posting.get('replies') or []
        for child in children:
            process_posting(child, depth + 1, record['comment_id'])
    
    # Handle different top-level structures
    if isinstance(raw, list):
        postings = raw
    elif isinstance(raw, dict):
        # Try common wrapper keys
        postings = (raw.get('postings') or raw.get('comments') or 
                    raw.get('data') or raw.get('items') or [])
        if isinstance(postings, dict):
            postings = list(postings.values())
    else:
        postings = []
    
    for posting in postings:
        process_posting(posting)
    
    return records


print('Helper functions defined.')

Helper functions defined.


## Step 1 — Explore the API response structure

**Run this cell first on one article** to see the actual JSON field names.
The `parse_comments()` function tries common field names but the actual
structure needs to be confirmed. Update the function if needed.

In [20]:
# ── Explore one article's API response ────────────────────────────────────────
test_url = ARTICLE_URLS[0]
test_id = extract_article_id(test_url)
print(f'Testing article ID: {test_id}')
print(f'API URL: https://apps.derstandard.at/forum/1/{test_id}')

result = fetch_comments(test_id)

if not result['ok']:
    print(f'ERROR: {result["error"]}')
else:
    raw = result['data']
    print(f'\nResponse type: {type(raw)}')
    
    if isinstance(raw, dict):
        print(f'Top-level keys: {list(raw.keys())}')
        # Show first posting to understand structure
        for key, val in raw.items():
            if isinstance(val, list) and len(val) > 0:
                print(f'\nKey "{key}" is a list of {len(val)} items.')
                print(f'First item keys: {list(val[0].keys()) if isinstance(val[0], dict) else type(val[0])}')
                print(f'First item sample:')
                print(json.dumps(val[0], indent=2, ensure_ascii=False)[:1500])
                break
    elif isinstance(raw, list):
        print(f'Response is a list of {len(raw)} items')
        if raw:
            print(f'First item keys: {list(raw[0].keys()) if isinstance(raw[0], dict) else type(raw[0])}')
            print(f'\nFirst item sample:')
            print(json.dumps(raw[0], indent=2, ensure_ascii=False)[:1500])

Testing article ID: 2000129268290
API URL: https://apps.derstandard.at/forum/1/2000129268290


ERROR: HTTP 404


In [21]:
# ── After seeing the structure above, update field names here if needed ────────
# Example: if text is under 'message' not 'text', add it to the parse function.
# This cell is just a reminder — edit parse_comments() directly.

# Quick test of parsing on the sample:
if result['ok']:
    sample_records = parse_comments(result['data'], test_id, test_url)
    print(f'Parsed {len(sample_records)} comments from test article')
    if sample_records:
        print('\nFirst parsed record:')
        print(json.dumps(sample_records[0], indent=2, ensure_ascii=False, default=str))
        
        # Check how many have non-empty text
        with_text = [r for r in sample_records if r.get('text', '').strip()]
        print(f'\nRecords with text: {len(with_text)}/{len(sample_records)}')
        
        # Show a few comment texts
        print('\nSample comment texts:')
        for r in with_text[:3]:
            print(f'  [{r["depth"]}] {str(r["text"])[:200]}')
            print()

## Step 2 — Full scrape across all articles

In [22]:
all_comments = []
scrape_log = []

for i, url in enumerate(ARTICLE_URLS):
    article_id = extract_article_id(url)
    if not article_id:
        print(f'[{i+1}/{len(ARTICLE_URLS)}] SKIP — could not extract ID from: {url}')
        continue
    
    print(f'[{i+1}/{len(ARTICLE_URLS)}] Fetching article {article_id}...', end=' ')
    
    result = fetch_comments(article_id)
    
    log_entry = {
        'article_id': article_id,
        'article_url': url,
        'scraped_at': datetime.utcnow().isoformat(),
        'ok': result['ok'],
        'error': result.get('error'),
        'n_comments': 0,
    }
    
    if result['ok']:
        comments = parse_comments(result['data'], article_id, url)
        log_entry['n_comments'] = len(comments)
        all_comments.extend(comments)
        print(f'{len(comments)} comments')
    else:
        print(f'ERROR — {result["error"]}')
    
    scrape_log.append(log_entry)
    
    # Polite rate limiting
    if i < len(ARTICLE_URLS) - 1:
        time.sleep(RATE_LIMIT_SECONDS)

print(f'\nDone. Total comments collected: {len(all_comments)}')
print(f'Articles succeeded: {sum(1 for l in scrape_log if l["ok"])}/{len(scrape_log)}')

[1/6] Fetching article 2000129268290... ERROR — HTTP 404
[2/6] Fetching article 2000123917902... ERROR — HTTP 404
[3/6] Fetching article 2000092116447... ERROR — HTTP 404
[4/6] Fetching article 2000142962434... ERROR — HTTP 404
[5/6] Fetching article 3000000319323... ERROR — HTTP 404
[6/6] Fetching article 2000135466855... ERROR — HTTP 404

Done. Total comments collected: 0
Articles succeeded: 0/6


In [23]:
# ── Save results ───────────────────────────────────────────────────────────────
output_path = DATA_DIR / 'derstandard_comments.json'
log_path = DATA_DIR / 'derstandard_scrape_log.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_comments, f, ensure_ascii=False, indent=2, default=str)

with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(scrape_log, f, ensure_ascii=False, indent=2, default=str)

print(f'Comments saved to: {output_path}')
print(f'Scrape log saved to: {log_path}')

Comments saved to: ../data/derstandard_comments.json
Scrape log saved to: ../data/derstandard_scrape_log.json


## Step 3 — Basic QA on collected data

In [24]:
df = pd.DataFrame(all_comments)
print(f'Total records: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print()

# Comments per article
print('Comments per article:')
print(df.groupby('article_id').size().sort_values(ascending=False).to_string())
print()

# Text coverage
has_text = df['text'].apply(lambda x: bool(str(x).strip()) if x else False)
print(f'Records with non-empty text: {has_text.sum()}/{len(df)} ({100*has_text.mean():.1f}%)')
print()

# Depth distribution (thread structure)
if 'depth' in df.columns:
    print('Comment depth distribution:')
    print(df['depth'].value_counts().sort_index().to_string())

Total records: 0
Columns: []

Comments per article:


KeyError: 'article_id'

In [ ]:
# Keyword check — does the scraped text contain your research terms?
KEYWORDS = ['Kindeswohl', 'Entfremdung', 'Sorgerecht', 'Obsorge', 'Umgang', 
            'Elternteil', 'Parental Alienation', 'PAS']

text_series = df['text'].fillna('').astype(str)

print('Keyword hit counts across all comments:')
for kw in KEYWORDS:
    hits = text_series.str.contains(kw, case=False, na=False).sum()
    print(f'  {kw:<25} {hits:>4} comments')

In [ ]:
# Show a sample of keyword-matching comments for sanity check
SAMPLE_KEYWORD = 'Kindeswohl'

matches = df[text_series.str.contains(SAMPLE_KEYWORD, case=False, na=False)]
print(f'Sample comments containing "{SAMPLE_KEYWORD}":\n')
for _, row in matches.head(5).iterrows():
    print(f'Article: {row["article_id"]} | Depth: {row.get("depth", "?")} | Upvotes: {row.get("upvotes", "?")}')  
    print(str(row['text'])[:400])
    print('---')

## Notes on parse_comments() — what to fix if text is empty

If the QA shows 0 records with text, the field names in `parse_comments()` 
don't match the actual API response. Go back to the Step 1 exploration cell 
and look at what field actually contains the comment text, then update these 
lines in `parse_comments()`:

```python
'text': posting.get('text') or posting.get('body') or posting.get('content', ''),
'comment_id': posting.get('id') or posting.get('postingId'),
'timestamp': posting.get('publishedDate') or posting.get('createdAt') or posting.get('date', ''),
```

The exploration cell prints the actual field names and a sample record —
that's your reference.